# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
# imports


import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [ ]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [ ]:
# set up environment
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    


In [ ]:
system_prompt = """
You are a helpful assistant that explains technical concepts with a very simple language.
"""


In [ ]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)
"""

In [ ]:
# Get gpt-4o-mini to answer, with streaming
def my_assistant(question, model=MODEL_GPT):
    """
    This function takes a question and returns a response from the assistant.
    """

    if model == MODEL_LLAMA:
        model = "llama3.2"
        OLLAMA_BASE_URL = "http://localhost:11434/v1"
        ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
        stream = ollama.chat.completions.create(
            model = model ,
            messages = [    
                {"role":"system","content":system_prompt},
                {"role": "user", "content": question},
            ],
            stream = True
        )
    else:
        model = MODEL_GPT
        openai = OpenAI()
        stream = openai.chat.completions.create(
            model = model ,
            messages = [
                {"role":"system","content":system_prompt},
                {"role": "user", "content": question},
            ],
            stream = True

        )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)
    return response


In [ ]:
my_assistant(question, model=MODEL_GPT)

In [ ]:
# Get Llama 3.2 to answer
my_assistant(question, model=MODEL_LLAMA)

Call GPT mdoel to compare both models performance

In [ ]:
def comapre_answers(question, model1=MODEL_GPT, model2=MODEL_LLAMA):
    """
    This function takes a question and returns a response from the assistant.
    """
    response1 = my_assistant(question, model=model1)
    response2 = my_assistant(question, model=model2)
    system_prompt = """
    You are a helpful assistant that compares the answers of two models.
    You will be given the answer of two models and you will need to compare them.
    You will need to return the answer of the model that is more helpful.
    You will need to return the answer of the model that is more accurate.
    You will need to return the answer of the model that is more concise.
    You will need to return the answer of the model that is more clear.
    You will need to return the answer of the model that is more detailed.
    You will need to return the answer of the model that is more easy to understand.
    You will need to return the answer of the model that is more easy to follow.
    """
    openai = OpenAI()
    response = openai.chat.completions.create(
        model = MODEL_GPT,
        messages = [
            {"role":"system","content":system_prompt},
            {"role": "user", "content": f"Answer from {model1}: {response1}\nAnswer from {model2}: {response2}"},
        ],
    )
    return response.choices[0].message.content

In [ ]:
comapre_answers(question, model1=MODEL_GPT, model2=MODEL_LLAMA)